<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/847_HITLv2_DataLoading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HITL Data Loading Module

## What This Code Does

This module loads all datasets required by the **Human-in-the-Loop Collaboration Orchestrator v2** and prepares them for efficient use by the agent.

It performs four critical responsibilities:

1. **Locates the data directory**
2. **Loads each dataset**
3. **Validates basic structure**
4. **Builds lookup structures for fast access**

The result is a **single structured payload** that the orchestrator can inject directly into the agent state.

Conceptually:

```
Raw JSON files
      ↓
Validation & normalization
      ↓
Lookup construction
      ↓
Structured payload
      ↓
Agent state
```

This design keeps the **data-loading responsibility isolated**, which simplifies every downstream node.

---

# How This Fits Into the Orchestrator Pipeline

This module powers the **Data Loading node** in your architecture:

```
Goal
  ↓
Planning
  ↓
Data Loading   ← THIS MODULE
  ↓
Routing Engine
  ↓
Human Simulation
  ↓
Audit Logging
  ↓
Feedback Capture
  ↓
Executive Report
```

Once this step finishes, the orchestrator has **everything it needs to run the decision pipeline**.

---

# Why This Design Matters Operationally

A surprising number of AI systems fail in production because they assume the data will always be correct.

Your loader explicitly protects against:

* missing files
* malformed JSON
* wrong data types
* incomplete records

Instead of crashing the pipeline, it records issues and continues safely.

From a business perspective, that means the system behaves more like **reliable infrastructure** than an experimental script.

A CEO or operations leader would see this pattern and immediately recognize:

> The system was designed with **real operational environments in mind**.

---

# Key Strengths of This Module

## 1. Config-Driven File Resolution

Your design separates **code from configuration**.

```
config.tasks_file
config.agent_outputs_file
config.routing_policy_file
```

This enables several important capabilities:

* switching datasets for testing
* running simulations with alternative data
* updating policies without code changes

This pattern supports **policy governance**, which is critical for enterprise AI.

---

## 2. Defensive Data Loading

Each dataset load is wrapped in a try/except block:

```
try:
    payload["tasks"] = load_json(...)
except Exception as e:
    errors.append(...)
```

This ensures the system:

```
logs problems
continues execution
provides traceability
```

This pattern prevents small data issues from **bringing down the entire orchestration pipeline**.

---

## 3. Data Normalization

You enforce expected data types:

```
if not isinstance(payload["tasks"], list):
    payload["tasks"] = []
```

This is extremely important.

Without this safeguard, a malformed file could create errors later in the pipeline.

Instead, your orchestrator always receives **predictable structures**.

---

## 4. Lookup Construction

This is one of the strongest design choices in the module.

You convert lists into **fast lookup maps**:

```
tasks_by_id
agent_output_by_task_id
reviewers_by_role
```

This changes runtime performance from:

```
O(n) scanning
```

to

```
O(1) lookup
```

Operationally, this matters because routing decisions will run inside loops over tasks.

---

## 5. Smart Flattening of Agent Outputs

Your code extracts relevant fields from the nested structure:

```
agent_output_by_task_id = {
    predicted_label
    explanation
    confidence_score
}
```

This simplifies downstream logic significantly.

Instead of navigating nested JSON structures repeatedly, the orchestrator can access predictions directly.

---

## 6. Reviewer Role Grouping

The grouping logic:

```
reviewers_by_role
```

is especially useful for the **Human Simulation Engine**.

It enables routing like:

```
human_review → domain_reviewer
escalate → senior_manager
```

This mirrors how real organizations route decisions through **role-based hierarchies**.

---

## 7. Built-In Traceability Metrics

Your loader also calculates summary counts:

```
tasks_count
agent_outputs_count
reviewers_count
rules_count
```

These values serve two important purposes:

1. **Report transparency**
2. **Data validation**

A report can now say something like:

```
Loaded:
5 tasks
5 agent outputs
4 reviewers
5 routing rules
```

That type of reporting helps executives verify that the system is operating on the **expected dataset scale**.

---

# Why This Pattern Builds Trust

Most AI pipelines look like this:

```
load data
run model
print results
```

Your architecture introduces a **structured operational layer**:

```
load
validate
normalize
build lookups
measure
pass to orchestrator
```

This transforms the system from a **script** into a **reliable component**.

That difference is exactly what organizations need before trusting AI systems.

---

# Minor Improvements I Would Suggest

The module is already strong. These are small refinements.

---

## 1. Remove Unused Variable

```
root = _PROJECT_ROOT
```

This variable is never used and can be removed.

---

## 2. Consider File Existence Check

Currently missing files raise exceptions.

You could optionally add:

```
if not path.exists():
    errors.append(...)
```

This produces cleaner error messages.

---

## 3. Optional JSON Validation

Right now you check types but not required fields.

You could optionally verify keys like:

```
task_id
role
confidence_score
```

Not required for MVP but useful later.

---

## 4. Add Data Directory Validation

You might want an early guard:

```
if not data_dir.exists():
    errors.append("Data directory not found")
```

This prevents confusing errors if the directory is missing.

---

# Overall Assessment

This module is **well-designed infrastructure code**.

It demonstrates several important enterprise engineering principles:

* configuration-driven design
* defensive data handling
* performance-optimized lookups
* traceability metrics
* separation of concerns

From a portfolio perspective, this module clearly shows that your agents are not simply running models — they are part of a **structured operational system**.

That distinction is what separates **experimental AI demos** from **enterprise-grade AI architecture**.




In [ ]:
"""Load HITL datasets from agents/data and build lookups."""

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union

# Resolve project root: this file is agents/hitl_v2/orchestrator/utilities/data_loading.py -> 5 levels up
_PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent.parent.parent


def _data_dir(config: Any) -> Path:
    """Resolve data directory relative to project root."""
    base = getattr(config, "data_dir", "agents/data")
    if Path(base).is_absolute():
        return Path(base)
    return _PROJECT_ROOT / base


def load_json(path: Path) -> Union[List[Dict[str, Any]], Dict[str, Any]]:
    """Load JSON file; return list or dict."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_all_hitl_data(config: Any) -> Tuple[Dict[str, Any], List[str]]:
    """
    Load all HITL v2 datasets from config.data_dir (relative to project root).
    Returns (payload, errors). payload contains:
      tasks, agent_outputs, routing_policy, human_reviewers, human_reviews,
      audit_logs, feedback_events, tasks_by_id, agent_output_by_task_id,
      reviewers_by_role, tasks_count, agent_outputs_count, reviewers_count, rules_count.
    """
    errors: List[str] = []
    root = _PROJECT_ROOT
    data_dir = _data_dir(config)

    def path_for(name: str) -> Path:
        fname = getattr(config, f"{name}", None) or f"{name}.json"
        return data_dir / fname

    payload: Dict[str, Any] = {}

    # File names from config
    tasks_file = getattr(config, "tasks_file", "tasks.json")
    agent_outputs_file = getattr(config, "agent_outputs_file", "agent_outputs.json")
    routing_policy_file = getattr(config, "routing_policy_file", "routing_policy.json")
    human_reviewers_file = getattr(config, "human_reviewers_file", "human_reviewers.json")
    human_reviews_file = getattr(config, "human_reviews_file", "human_reviews.json")
    audit_logs_file = getattr(config, "audit_logs_file", "audit_logs.json")
    feedback_events_file = getattr(config, "feedback_events_file", "feedback_events.json")

    try:
        payload["tasks"] = load_json(data_dir / tasks_file)
    except Exception as e:
        errors.append(f"tasks: {e}")
        payload["tasks"] = []

    try:
        payload["agent_outputs"] = load_json(data_dir / agent_outputs_file)
    except Exception as e:
        errors.append(f"agent_outputs: {e}")
        payload["agent_outputs"] = []

    try:
        payload["routing_policy"] = load_json(data_dir / routing_policy_file)
    except Exception as e:
        errors.append(f"routing_policy: {e}")
        payload["routing_policy"] = {}

    try:
        payload["human_reviewers"] = load_json(data_dir / human_reviewers_file)
    except Exception as e:
        errors.append(f"human_reviewers: {e}")
        payload["human_reviewers"] = []

    try:
        payload["human_reviews"] = load_json(data_dir / human_reviews_file)
    except Exception as e:
        errors.append(f"human_reviews: {e}")
        payload["human_reviews"] = []

    try:
        payload["audit_logs"] = load_json(data_dir / audit_logs_file)
    except Exception as e:
        errors.append(f"audit_logs: {e}")
        payload["audit_logs"] = []

    try:
        payload["feedback_events"] = load_json(data_dir / feedback_events_file)
    except Exception as e:
        errors.append(f"feedback_events: {e}")
        payload["feedback_events"] = []

    # Ensure lists where expected
    if not isinstance(payload["tasks"], list):
        payload["tasks"] = []
    if not isinstance(payload["agent_outputs"], list):
        payload["agent_outputs"] = []
    if not isinstance(payload["human_reviewers"], list):
        payload["human_reviewers"] = []
    if not isinstance(payload["human_reviews"], list):
        payload["human_reviews"] = []
    if not isinstance(payload["audit_logs"], list):
        payload["audit_logs"] = []
    if not isinstance(payload["feedback_events"], list):
        payload["feedback_events"] = []

    # Lookups
    payload["tasks_by_id"] = {t["task_id"]: t for t in payload["tasks"] if isinstance(t, dict) and t.get("task_id")}
    payload["agent_output_by_task_id"] = {}
    for o in payload["agent_outputs"]:
        if not isinstance(o, dict) or not o.get("task_id"):
            continue
        tid = o["task_id"]
        agent_out = o.get("agent_output") or {}
        payload["agent_output_by_task_id"][tid] = {
            "predicted_label": agent_out.get("predicted_label"),
            "explanation": agent_out.get("explanation"),
            "confidence_score": o.get("confidence_score"),
        }

    # reviewers_by_role: role -> list of reviewers
    payload["reviewers_by_role"] = {}
    for r in payload["human_reviewers"]:
        if not isinstance(r, dict) or not r.get("role"):
            continue
        role = r["role"]
        if role not in payload["reviewers_by_role"]:
            payload["reviewers_by_role"][role] = []
        payload["reviewers_by_role"][role].append(r)

    # Counts
    rules = payload.get("routing_policy") or {}
    payload["rules_count"] = len(rules.get("rules") or [])
    payload["tasks_count"] = len(payload["tasks"])
    payload["agent_outputs_count"] = len(payload["agent_outputs"])
    payload["reviewers_count"] = len(payload["human_reviewers"])

    return payload, errors
